In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # E5.2 · Catálogo de Reglas DQ Silver
# MAGIC
# MAGIC Este notebook registra las reglas de calidad aplicables
# MAGIC a la capa Silver del proyecto YelpConnect.
# MAGIC
# MAGIC Principio:
# MAGIC
# MAGIC - Bronze conserva.
# MAGIC - Silver conforma.
# MAGIC - DQ Silver valida que los datos sean confiables.
# MAGIC - Trust & Safety evalúa señales de riesgo posteriormente.
# MAGIC
# MAGIC Las reglas se almacenan como datos en:
# MAGIC
# MAGIC `workspace.yelp_gov.dq_reglas`


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

GOV_SCHEMA = "yelp_gov"

DQ_RULE_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "dq_reglas"
)


# COMMAND ----------

# ============================================================
# 3. CREAR SCHEMA DE GOBIERNO
# ============================================================

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS
{CATALOG}.{GOV_SCHEMA}
""")


print(
    f"✔ Schema disponible: "
    f"{CATALOG}.{GOV_SCHEMA}"
)


# COMMAND ----------

# ============================================================
# 4. CREAR TABLA DE REGLAS
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DQ_RULE_TABLE}
(
    regla_id       STRING,
    capa           STRING,
    tabla_objetivo STRING,
    columna        STRING,
    dimension      STRING,
    tipo_regla     STRING,
    descripcion    STRING,
    expresion_sql  STRING,
    criticidad     STRING,
    umbral_pct     DOUBLE
)
USING DELTA
""")


print(
    f"✔ Tabla disponible: "
    f"{DQ_RULE_TABLE}"
)

# COMMAND ----------

# ============================================================
# 4.1. AJUSTAR ESQUEMA dq_reglas
# ============================================================

DQ_RULE_TABLE = "workspace.yelp_gov.dq_reglas"

columnas_actuales = {
    campo.name
    for campo in spark.table(DQ_RULE_TABLE).schema.fields
}

columnas_faltantes = []

if "capa" not in columnas_actuales:
    columnas_faltantes.append("capa STRING")

if "tipo_regla" not in columnas_actuales:
    columnas_faltantes.append("tipo_regla STRING")


if columnas_faltantes:

    columnas_sql = ", ".join(
        columnas_faltantes
    )

    spark.sql(f"""
        ALTER TABLE {DQ_RULE_TABLE}
        ADD COLUMNS (
            {columnas_sql}
        )
    """)

    print(
        f"✔ Columnas agregadas: "
        f"{columnas_sql}"
    )

else:

    print(
        "✔ El esquema de dq_reglas ya está actualizado."
    )


# Validación
display(
    spark.sql(f"""
        DESCRIBE TABLE {DQ_RULE_TABLE}
    """)
)

# COMMAND ----------

# ============================================================
# 5. SCHEMA EXPLÍCITO DEL CATÁLOGO
# ============================================================

schema_reglas = StructType([

    StructField(
        "regla_id",
        StringType(),
        False
    ),

    StructField(
        "capa",
        StringType(),
        False
    ),

    StructField(
        "tabla_objetivo",
        StringType(),
        False
    ),

    StructField(
        "columna",
        StringType(),
        True
    ),

    StructField(
        "dimension",
        StringType(),
        False
    ),

    StructField(
        "tipo_regla",
        StringType(),
        False
    ),

    StructField(
        "descripcion",
        StringType(),
        False
    ),

    StructField(
        "expresion_sql",
        StringType(),
        True
    ),

    StructField(
        "criticidad",
        StringType(),
        False
    ),

    StructField(
        "umbral_pct",
        DoubleType(),
        False
    )
])


# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Reglas Silver
# MAGIC
# MAGIC Se definen reglas para:
# MAGIC
# MAGIC - Review
# MAGIC - Business
# MAGIC - User
# MAGIC - Tip
# MAGIC - Check-in


# COMMAND ----------

# ============================================================
# 6. CATÁLOGO DE REGLAS SILVER
# ============================================================

reglas_silver = [

    # ========================================================
    # REVIEW
    # ========================================================

    (
        "DQ-SL-001",
        "silver",
        "workspace.yelp_silver.slv_review",
        "stars",
        "validez",
        "CONDITION",
        "El rating de una reseña debe estar entre 1 y 5",
        "stars BETWEEN 1 AND 5",
        "alta",
        100.0
    ),

    (
        "DQ-SL-002",
        "silver",
        "workspace.yelp_silver.slv_review",
        "review_id",
        "unicidad",
        "UNIQUENESS",
        "review_id debe ser único dentro de Silver",
        None,
        "alta",
        100.0
    ),

    (
        "DQ-SL-003",
        "silver",
        "workspace.yelp_silver.slv_review",
        "business_id",
        "integridad",
        "CONDITION",
        "Toda Review debe referenciar un Business existente",
        """
        business_id IN (
            SELECT business_id
            FROM workspace.yelp_silver.slv_business
        )
        """,
        "alta",
        100.0
    ),

    (
        "DQ-SL-004",
        "silver",
        "workspace.yelp_silver.slv_review",
        "review_id",
        "completitud",
        "CONDITION",
        "review_id no debe ser nulo",
        "review_id IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-005",
        "silver",
        "workspace.yelp_silver.slv_review",
        "business_id",
        "completitud",
        "CONDITION",
        "business_id no debe ser nulo",
        "business_id IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-006",
        "silver",
        "workspace.yelp_silver.slv_review",
        "user_id",
        "completitud",
        "CONDITION",
        "user_id no debe ser nulo",
        "user_id IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-007",
        "silver",
        "workspace.yelp_silver.slv_review",
        "review_text",
        "completitud",
        "CONDITION",
        "El texto de la reseña no debe ser nulo ni vacío",
        """
        review_text IS NOT NULL
        AND LENGTH(TRIM(review_text)) > 0
        """,
        "media",
        99.0
    ),

    (
        "DQ-SL-008",
        "silver",
        "workspace.yelp_silver.slv_review",
        "review_ts",
        "validez",
        "CONDITION",
        "La fecha de la reseña debe poder representarse como timestamp",
        "review_ts IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-009",
        "silver",
        "workspace.yelp_silver.slv_review",
        "useful",
        "consistencia",
        "CONDITION",
        "El contador useful no puede ser negativo",
        "useful IS NULL OR useful >= 0",
        "media",
        100.0
    ),

    (
        "DQ-SL-010",
        "silver",
        "workspace.yelp_silver.slv_review",
        "funny",
        "consistencia",
        "CONDITION",
        "El contador funny no puede ser negativo",
        "funny IS NULL OR funny >= 0",
        "media",
        100.0
    ),

    (
        "DQ-SL-011",
        "silver",
        "workspace.yelp_silver.slv_review",
        "cool",
        "consistencia",
        "CONDITION",
        "El contador cool no puede ser negativo",
        "cool IS NULL OR cool >= 0",
        "media",
        100.0
    ),

    (
        "DQ-SL-012",
        "silver",
        "workspace.yelp_silver.slv_review",
        "user_id",
        "integridad",
        "CONDITION",
        "El user_id de Review debería estar presente en la muestra User",
        """
        user_id IN (
            SELECT user_id
            FROM workspace.yelp_silver.slv_user
        )
        """,
        "media",
        90.0
    ),

    (
        "DQ-SL-013",
        "silver",
        "workspace.yelp_silver.slv_review",
        "review_ts",
        "actualidad",
        "CONDITION",
        "La fecha de la reseña no puede ser posterior a su procesamiento Silver",
        """
        review_ts <= _silver_processed_at
        """,
        "media",
        100.0
    ),


    # ========================================================
    # BUSINESS
    # ========================================================

    (
        "DQ-SL-014",
        "silver",
        "workspace.yelp_silver.slv_business",
        "business_id",
        "completitud",
        "CONDITION",
        "business_id no debe ser nulo",
        "business_id IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-015",
        "silver",
        "workspace.yelp_silver.slv_business",
        "business_id",
        "unicidad",
        "UNIQUENESS",
        "business_id debe ser único",
        None,
        "alta",
        100.0
    ),

    (
        "DQ-SL-016",
        "silver",
        "workspace.yelp_silver.slv_business",
        "estrellas",
        "validez",
        "CONDITION",
        "El rating agregado del Business debe estar entre 0 y 5",
        """
        estrellas IS NULL
        OR estrellas BETWEEN 0 AND 5
        """,
        "media",
        100.0
    ),

    (
        "DQ-SL-017",
        "silver",
        "workspace.yelp_silver.slv_business",
        "n_resenas",
        "consistencia",
        "CONDITION",
        "El número de reseñas del Business no puede ser negativo",
        """
        n_resenas IS NULL
        OR n_resenas >= 0
        """,
        "media",
        100.0
    ),


    # ========================================================
    # USER
    # ========================================================

    (
        "DQ-SL-018",
        "silver",
        "workspace.yelp_silver.slv_user",
        "user_id",
        "completitud",
        "CONDITION",
        "user_id no debe ser nulo",
        "user_id IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-019",
        "silver",
        "workspace.yelp_silver.slv_user",
        "user_id",
        "unicidad",
        "UNIQUENESS",
        "user_id debe ser único",
        None,
        "alta",
        100.0
    ),


    # ========================================================
    # TIP
    # ========================================================

    (
        "DQ-SL-020",
        "silver",
        "workspace.yelp_silver.slv_tip",
        "business_id",
        "integridad",
        "CONDITION",
        "Todo Tip debe referenciar un Business existente",
        """
        business_id IN (
            SELECT business_id
            FROM workspace.yelp_silver.slv_business
        )
        """,
        "media",
        100.0
    ),


    # ========================================================
    # CHECK-IN
    # ========================================================

    (
        "DQ-SL-021",
        "silver",
        "workspace.yelp_silver.slv_checkin_event",
        "business_id",
        "integridad",
        "CONDITION",
        "Todo Check-in debe referenciar un Business existente",
        """
        business_id IN (
            SELECT business_id
            FROM workspace.yelp_silver.slv_business
        )
        """,
        "media",
        100.0
    ),

    (
        "DQ-SL-022",
        "silver",
        "workspace.yelp_silver.slv_checkin_event",
        "checkin_ts",
        "validez",
        "CONDITION",
        "Todo evento Check-in debe poseer un timestamp válido",
        "checkin_ts IS NOT NULL",
        "media",
        100.0
    )
]


# COMMAND ----------

# ============================================================
# 7. CREAR DATAFRAME
# ============================================================

df_reglas_silver = (
    spark.createDataFrame(
        reglas_silver,
        schema=schema_reglas
    )
)


print(
    f"Reglas Silver definidas: "
    f"{df_reglas_silver.count()}"
)


display(
    df_reglas_silver
    .orderBy(
        "regla_id"
    )
)


# COMMAND ----------

# ============================================================
# 8. CREAR VISTA TEMPORAL
# ============================================================

df_reglas_silver.createOrReplaceTempView(
    "src_dq_reglas_silver"
)


# COMMAND ----------

# ============================================================
# 9. MERGE IDEMPOTENTE
# ============================================================

spark.sql(f"""
MERGE INTO {DQ_RULE_TABLE} AS target

USING src_dq_reglas_silver AS source

ON target.regla_id = source.regla_id

WHEN MATCHED THEN
    UPDATE SET
        target.capa = source.capa,
        target.tabla_objetivo = source.tabla_objetivo,
        target.columna = source.columna,
        target.dimension = source.dimension,
        target.tipo_regla = source.tipo_regla,
        target.descripcion = source.descripcion,
        target.expresion_sql = source.expresion_sql,
        target.criticidad = source.criticidad,
        target.umbral_pct = source.umbral_pct

WHEN NOT MATCHED THEN
    INSERT
    (
        regla_id,
        capa,
        tabla_objetivo,
        columna,
        dimension,
        tipo_regla,
        descripcion,
        expresion_sql,
        criticidad,
        umbral_pct
    )

    VALUES
    (
        source.regla_id,
        source.capa,
        source.tabla_objetivo,
        source.columna,
        source.dimension,
        source.tipo_regla,
        source.descripcion,
        source.expresion_sql,
        source.criticidad,
        source.umbral_pct
    )
""")


print(
    "✔ Reglas Silver registradas/actualizadas."
)


# COMMAND ----------

# ============================================================
# 10. VALIDACIÓN
# ============================================================

total_silver = (
    spark.table(
        DQ_RULE_TABLE
    )
    .filter(
        "capa = 'silver'"
    )
    .count()
)


print("=" * 70)

print(
    "CATÁLOGO DATA QUALITY - SILVER"
)

print("=" * 70)

print(
    f"✔ Total reglas Silver: "
    f"{total_silver}"
)

print("=" * 70)


display(
    spark.sql(f"""
        SELECT

            regla_id,
            tabla_objetivo,
            columna,
            dimension,
            tipo_regla,
            criticidad,
            umbral_pct,
            descripcion

        FROM
            {DQ_RULE_TABLE}

        WHERE
            capa = 'silver'

        ORDER BY
            regla_id
    """)
)


# COMMAND ----------

# ============================================================
# 11. RESUMEN POR DIMENSIÓN
# ============================================================

display(
    spark.sql(f"""
        SELECT

            dimension,

            COUNT(*) AS total_reglas

        FROM
            {DQ_RULE_TABLE}

        WHERE
            capa = 'silver'

        GROUP BY
            dimension

        ORDER BY
            dimension
    """)
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Resultado esperado
# MAGIC
# MAGIC Deben existir **22 reglas Silver**.
# MAGIC
# MAGIC Las dimensiones cubiertas son:
# MAGIC
# MAGIC - Completitud
# MAGIC - Unicidad
# MAGIC - Validez
# MAGIC - Consistencia
# MAGIC - Integridad
# MAGIC - Actualidad